# CellLineSelector — Data Harmonisation Pipeline

Harmonises 15 cleaned source tables (DepMap/CCLE, Cellosaurus, GEO, HPA) onto a
single canonical key, `model_id` (ACH), and persists the result to DuckDB.

## Outputs

| table | grain | purpose |
|---|---|---|
| `cell_line_connection` | one row per ACH | identity hub — every alternative id as a list |
| `gene` | one row per ENSG | gene identity hub — names, symbols, uniprot ids |
| `coverage_matrix` | one row per ACH | per-line modality coverage |
| 15 source tables | as loaded | harmonised, `model_id` attached |

## Guiding principles

- **Flag, don't drop** — ambiguous identities are retained and marked, never silently resolved
- **Credit both** — a row mapping to 2 candidate ACHs is duplicated, one row each
- **Grain before joining** — event tables aggregated before any join
- **Identity hubs hold lists** — one row per entity, alternative ids as list columns,
  so joins stay one-to-one and no count is ever inflated by duplication

## Flow

```
load -> structural normalisation -> identity hub -> attach model_id
     -> resolve ambiguity -> coverage audit -> persist -> dimension tables -> verify
```

Author: Tee — CellLineSelector (AstraZeneca collaboration, 2025)

## 0. Configuration

Every constant used downstream is declared here, so nothing is redefined mid-notebook.

In [ ]:
import sys
import os
import re
import numpy as np
import pandas as pd
import duckdb

# resolve scripts dir relative to this notebook (src/pipeline -> src/scripts)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "scripts")))
from data_utils import load_clean_parquets

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# Everything this notebook writes lands in src/pipeline/outputs/, alongside the
# rest of the pipeline artefacts. DB_PATH is the DuckDB warehouse; the parquet
# exports in Section 9 keep Stage 1's file-based input contract intact.
OUT_DIR = os.path.abspath(os.path.join(os.getcwd(), "outputs"))
os.makedirs(OUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUT_DIR, "celllineselector.db")

# Containers that model_id may arrive wrapped in, before flattening
LIST_LIKE = (list, tuple, set, frozenset, np.ndarray, pd.Series, pd.Index)

# MODALITY = measured evidence. ANNOTATION = identity/metadata, which describes
# a line rather than assaying it. The split matters for coverage counting:
# sample_info is ~100% by construction and would inflate every line's score.
MODALITY_LAYERS   = ["depmap_expr", "geo_expr", "hpa_rna", "mutations", "fusions",
                     "proteomics", "metabolomics", "mirna", "signatures"]
ANNOTATION_LAYERS = ["sample_info", "cellosaurus", "hpa_desc", "depmap_profiles", "geo_info"]
ALL_LAYERS        = MODALITY_LAYERS + ANNOTATION_LAYERS

ENSG_PATTERN = r'^ensg\d+'

# Placeholder tokens meaning 'no value'. '.' is DepMap's null marker in
# fusions and must never be treated as a gene name.
NULL_TOKENS = ("", ".", "na", "n/a", "null", "none", "-")

print("OUT_DIR:", OUT_DIR)
print("DB_PATH:", DB_PATH)


## 1. Shared helpers

One definition each, all in this cell. Anything used more than once lives here
rather than being redefined at the point of use.

In [ ]:
def lower_all(df):
    """Lowercase and strip column headers and all string values."""
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].astype("string").str.strip().str.lower()
    return df


def normalise_accession(s, drop_prefix=None):
    """
    Normalise any CVCL-style identifier to bare 'cvcl_xxxx'.
    Sources write the same accession as 'CVCL_2063', 'cvcl:2063' or
    'RRID:CVCL_2063'; string matching is exact, so they must be unified
    before any join.
    """
    s = s.astype("string").str.strip()
    if drop_prefix:
        s = s.str.replace(rf"^{drop_prefix}[:_\-\s]*", "", regex=True)
    return (s.str.replace(r"^cvcl[:_\-\s]*", "cvcl_", regex=True)
             .str.replace(r"[^a-z0-9_]+$", "", regex=True))


def explode_ids(series):
    """
    Flatten a Series holding scalars, list-likes or nested list-likes into one
    flat Series. Bounded at 10 passes so pathological nesting fails loudly
    rather than looping forever.
    """
    s = series.dropna()
    for _ in range(10):
        if not s.map(lambda x: isinstance(x, LIST_LIKE)).any():
            break
        s = s.explode().dropna()
    return s


def build_lookup(roster, list_col, key_col="model_id"):
    """
    element-of-`list_col` -> sorted list of model_ids.
    Returns a LIST even for single matches, so ambiguity is never hidden by
    the return type.
    """
    flat = roster[[key_col, list_col]].explode(list_col).dropna(subset=[list_col])
    return flat.groupby(list_col)[key_col].agg(lambda s: sorted(set(s)))


def union_lists(values):
    """Union of all list values in a Series, ignoring NaN."""
    out = set()
    for v in values.dropna():
        out.update(v)
    return sorted(out)


def collect(df, val, name, key="model_id"):
    """
    model_id -> sorted set of `val`, as a named Series, for attaching an
    identifier axis to the hub. Prints and returns None if the source column
    is absent, so a missing column surfaces here rather than as a silently
    absent column several cells later.
    """
    if key not in df.columns or val not in df.columns:
        missing = key if key not in df.columns else val
        print(f"  [skip] {name}: column '{missing}' not present")
        return None
    d = df[[key, val]].dropna()
    if d[key].map(lambda x: isinstance(x, LIST_LIKE)).any():
        d = d.explode(key).dropna(subset=[key])
    return d.groupby(key)[val].agg(lambda s: sorted(set(s.dropna()))).rename(name)


def ensure_key_column(df, key):
    """Guarantee `key` is a column, promoting it from the index if needed."""
    if key in df.columns:
        return df
    out = df.reset_index()
    return out.rename(columns={out.columns[0]: key})


def report_cardinality(df, left, right, label=""):
    """Print the cardinality of left <-> right in both directions."""
    l2r = df.dropna(subset=[left]).groupby(left)[right].nunique()
    r2l = df.dropna(subset=[right]).groupby(right)[left].nunique()
    print(f"--- {label or f'{left} <-> {right}'} ---")
    print(f"{left:>28s} with >1 {right}: {(l2r > 1).sum():5d} / {len(l2r)}  (max {l2r.max()})")
    print(f"{right:>28s} with >1 {left}: {(r2l > 1).sum():5d} / {len(r2l)}  (max {r2l.max()})")
    print(f"strictly 1:1: {(l2r.max() == 1) and (r2l.max() == 1)}\n")


def explode_model_id(df, name=""):
    """
    Flag ambiguity, then split list-valued model_id into one row per candidate ACH.

    Flagging MUST happen before the explode: afterwards the list is gone and a
    duplicated row is indistinguishable from an original.
    """
    if "model_id" not in df.columns:
        return df
    out = df.copy()
    out["n_model_id"] = out["model_id"].map(
        lambda x: len(x) if isinstance(x, LIST_LIKE) else (0 if pd.isna(x) else 1))
    out["is_ambiguous"] = out["n_model_id"] > 1
    before = len(out)
    out = out.explode("model_id").reset_index(drop=True)
    print(f"  {name:18s} {before:>9,} -> {len(out):>9,} rows  "
          f"(+{len(out) - before}, ambiguous {int(out['is_ambiguous'].sum())}, "
          f"unmatched {int((out['n_model_id'] == 0).sum())})")
    return out


def unwrap_model_id(df, name=""):
    """
    Convert model_id from list type to plain string. Run AFTER exploding — each
    row should hold one candidate, just still boxed in a length-1 list.
    Rows still holding >1 candidate are flagged, never silently resolved.

    This is what makes plain SQL joins work: DuckDB cannot join on a LIST column.
    """
    if df is None or "model_id" not in df.columns:
        return df
    out = df.copy()

    def unwrap(x):
        if isinstance(x, LIST_LIKE):
            if len(x) == 0:
                return np.nan
            if len(x) == 1:
                return x[0]
            return x
        return x

    out["model_id"] = out["model_id"].map(unwrap)
    n_still = int(out["model_id"].map(lambda x: isinstance(x, LIST_LIKE)).sum())
    out["model_id"] = out["model_id"].astype("string")
    flag = "  <-- explode this table first!" if n_still else ""
    print(f"  {name:18s} dtype {out['model_id'].dtype}, still-list rows: {n_still}{flag}")
    return out


def clean_ensg(s):
    """Lowercase, trim, and strip the Ensembl version suffix (.12)."""
    return (s.astype(str).str.strip().str.lower()
             .str.replace(r"\.\d+$", "", regex=True))

## 2. Load cleaned source tables

All 15 tables have passed through `lower_all()` and `normalize_cellname()` in
the upstream cleaning notebooks.

In [ ]:
tables = load_clean_parquets()

hpa_rna         = tables["hpa_rna"]          # HPA RNA expression (LONG: one row per gene per line)
hpa_desc        = tables["hpa_desc"]         # HPA cell line metadata
depmap_expr     = tables["depmap_expr"]      # DepMap expression (WIDE: gene per column)
geo_expr        = tables["geo_expr"]         # GEO expression (gene-rows, GSM-columns on load)
proteomics      = tables["proteomics"]       # CCLE proteomics (WIDE, UniProt-keyed, log-ratio)
protein_map     = tables["protein_map"]      # UniProt <-> gene symbol reference
fusions         = tables["fusions"]          # fusion events (two gene keys per row)
mutations       = tables["mutations"]        # mutation events (profile-keyed)
cellosaurus     = tables["cellosaurus"]      # identity reference
depmap_profiles = tables["depmap_profiles"]  # model <-> profile registry
sample_info     = tables["sample_info"]      # DepMap ACH roster
geo_info        = tables["geo_info"]         # GEO sample metadata
metabolomics    = tables["metabolomics"]     # CCLE metabolomics
mirna           = tables["mirna"]            # miRNA expression (miRNA-rows on load)
signatures      = tables["signatures"]       # cell line signatures/scores

print(f"loaded {len(tables)} tables")
for k, v in tables.items():
    print(f"  {k:18s} {v.shape}")

## 3. Structural normalisation

Source-shape work only — naming, formats, orientation, row and column filtering.
All of it happens before any identity logic, so later sections can assume a
consistent shape.

### 3.1 Canonical column naming

Six tables already carry an ACH under a different name. Renaming up front means
`model_id` is the only key referenced from here on — and avoids the trap of
joining on `depmap_id` early, then renaming late and breaking those joins.

In [ ]:
sample_info     = sample_info.rename(columns={"depmap_id": "model_id"})
depmap_profiles = depmap_profiles.rename(columns={"modelid": "model_id"})
fusions         = fusions.rename(columns={"modelid": "model_id"})
signatures      = signatures.rename(columns={"modelid": "model_id"})
proteomics      = proteomics.rename(columns={"depmap_id": "model_id"})
metabolomics    = metabolomics.rename(columns={"depmap_id": "model_id"})

for name, old in [("sample_info", "depmap_id"), ("depmap_profiles", "modelid"),
                  ("fusions", "modelid"), ("signatures", "modelid"),
                  ("proteomics", "depmap_id"), ("metabolomics", "depmap_id")]:
    print(f"{name:18s} {old} -> model_id")

### 3.2 Accession normalisation

`cellosaurus_accession` and `sample_info.rrid` are stripped to bare `cvcl_xxxx`
so they join on identical footing.

In [ ]:
cellosaurus = lower_all(cellosaurus)
sample_info = lower_all(sample_info)

cellosaurus["cellosaurus_accession"] = normalise_accession(cellosaurus["cellosaurus_accession"])
sample_info["rrid"] = normalise_accession(sample_info["rrid"], drop_prefix="rrid")

print("cellosaurus rows:", len(cellosaurus),
      "| missing accession:", cellosaurus["cellosaurus_accession"].isna().sum())
print("sample_info rows:", len(sample_info),
      "| missing rrid:     ", sample_info["rrid"].isna().sum())

### 3.3 Table orientation

`geo_expr` and `mirna` arrive feature-as-rows. Both are transposed to the target
form: one row per sample, one column per feature.

In [ ]:
geo_expr = geo_expr.set_index("gene").T
geo_expr.index.name = "sample"
geo_expr.columns.name = None
geo_expr = geo_expr.reset_index()

# 'description' is metadata, not a sample — dropped before transposing so it
# doesn't become a row
mirna = mirna.drop(columns=["description"]).set_index("name").T
mirna.index.name = "ccle_name"
mirna.columns.name = None
mirna = mirna.reset_index()

n_coerced = mirna.set_index("ccle_name").apply(pd.to_numeric, errors="coerce").isna().sum().sum()
print("geo_expr :", geo_expr.shape)
print("mirna    :", mirna.shape, f"| non-numeric cells: {n_coerced}")

### 3.4 Mutations — restrict to protein-coding

`vepbiotype` is VEP's annotation of the transcript a variant sits in. Downstream
analysis targets protein-altering consequences on canonical coding transcripts,
so non-coding biotypes are dropped once here rather than filtered repeatedly.

This filter also makes `mutations` usable as a protein-coding whitelist in 3.6.

In [ ]:
n_before = len(mutations)
mutations = mutations[mutations["vepbiotype"] == "protein_coding"].reset_index(drop=True)
print(f"vepbiotype filter: {n_before:,} -> {len(mutations):,} rows "
      f"({n_before - len(mutations):,} dropped)")

### 3.5 hpa_desc — split `patient` into gender and age

The raw format is inconsistent: some rows are `"male, 72"`, others age-only
(`"13"`). Both patterns are handled explicitly; rows matching neither are left
NaN and reported rather than guessed at.

In [ ]:
patient = hpa_desc["patient"].astype("string").str.strip()
gendered = patient.str.extract(r"^(?P<gender>\w+)\s*,\s*(?P<age>\d+)$")
age_only = patient.str.extract(r"^(?P<age>\d+)$")

hpa_desc["gender"] = gendered["gender"]
hpa_desc["age"] = pd.to_numeric(gendered["age"].combine_first(age_only["age"]), errors="coerce")

unparsed = (patient.notna() & (patient != "") & hpa_desc["age"].isna()).sum()
print(f"gender parsed: {hpa_desc['gender'].notna().sum()} | "
      f"age parsed: {hpa_desc['age'].notna().sum()} | unparsed: {unparsed}")

### 3.6 Restrict expression matrices to protein-coding genes

`depmap_expr` carries ~54,000 gene columns — the full transcriptome, including
pseudogenes, lincRNAs and antisense transcripts. Analysis targets protein-coding
genes, so the rest are dropped once here.

**The whitelist is inferred, not read from a biotype table.** It unions two proxies:

- `mutations`, already filtered to `vepbiotype == 'protein_coding'` — but it only
  lists genes that are coding **and mutated somewhere**, so it under-counts
- `hpa_rna`, which covers the coding transcriptome regardless of mutation status

The union lands near the ~19,500 protein-coding genes in the human genome, which
is the main evidence the inference is sound. A GENCODE biotype table would be the
rigorous source; this is a defensible substitute given what's loaded.

In [ ]:
def protein_coding_genes(mutations, hpa_rna=None, mut_col="ensemblgeneid",
                         hpa_col="gene", include_hpa=True):
    """Protein-coding ENSG whitelist, unioned from mutations and hpa_rna."""
    mut = {g for g in clean_ensg(mutations[mut_col].dropna()) if g.startswith("ensg")}
    out = set(mut)
    print(f"  from mutations: {len(mut):,}")
    if include_hpa and hpa_rna is not None:
        hpa = {g for g in clean_ensg(hpa_rna[hpa_col].dropna()) if g.startswith("ensg")}
        print(f"  from hpa_rna:   {len(hpa):,}  (+{len(hpa - mut):,} not in mutations)")
        out |= hpa
    print(f"  whitelist:      {len(out):,}")
    return out


def filter_wide_to_genes(df, keep_genes, name="",
                         id_cols=("model_id", "profileid", "sample",
                                  "n_model_id", "is_ambiguous")):
    """Drop gene columns outside the whitelist. Non-gene columns are untouched."""
    ids = [c for c in df.columns
           if c in id_cols or not str(c).lower().startswith("ensg")]
    gcols = [c for c in df.columns if str(c).lower().startswith("ensg")]
    keep = [c for c in gcols if str(c).lower() in keep_genes]
    out = df[ids + keep]
    print(f"  {name:14s} {len(gcols):>7,} gene cols -> {len(keep):>7,} kept "
          f"({len(gcols) - len(keep):,} dropped, {len(keep)/max(len(gcols),1)*100:.1f}% retained)")
    return out


coding_genes = protein_coding_genes(mutations, hpa_rna)
depmap_expr = filter_wide_to_genes(depmap_expr, coding_genes, "depmap_expr")
geo_expr    = filter_wide_to_genes(geo_expr,    coding_genes, "geo_expr")

### 3.7 Fusions — keep only fusions where BOTH partners are protein-coding

A fusion joins two genes. If either partner is non-coding — or a viral reference
accession — the event cannot be corroborated against expression, since those
genes were dropped from `depmap_expr` and `geo_expr` in 3.6.

The cross-tab is printed first so the loss is visible before it is applied.
Three states are distinguished, not two: `not_ensg` (viral accessions and other
non-gene ids) is separated from `non_coding`, because those are different things.

In [ ]:
def fusion_coding_breakdown(fusions, coding_genes,
                            col1="gene1_ens_id", col2="gene2_ens_id"):
    """Cross-tab of fusion partners by protein-coding status."""
    f = fusions.copy()
    g1, g2 = clean_ensg(f[col1]), clean_ensg(f[col2])

    def status(g):
        is_ensg = g.str.match(r"^ensg\d+", na=False)
        return pd.Series(np.where(~is_ensg, "not_ensg",
                         np.where(g.isin(coding_genes), "coding", "non_coding")),
                         index=g.index)

    f["g1_status"], f["g2_status"] = status(g1), status(g2)
    n = len(f)
    both    = ((f.g1_status == "coding") & (f.g2_status == "coding")).sum()
    neither = ((f.g1_status == "non_coding") & (f.g2_status == "non_coding")).sum()
    only_g1 = ((f.g1_status == "coding") & (f.g2_status == "non_coding")).sum()
    only_g2 = ((f.g1_status == "non_coding") & (f.g2_status == "coding")).sum()
    junk    = ((f.g1_status == "not_ensg") | (f.g2_status == "not_ensg")).sum()

    print(f"total fusion rows: {n:,}\n")
    print(f"  both coding:              {both:>8,}  ({both/n*100:5.1f}%)")
    print(f"  neither coding:           {neither:>8,}  ({neither/n*100:5.1f}%)")
    print(f"  gene1 coding, gene2 not:  {only_g1:>8,}  ({only_g1/n*100:5.1f}%)")
    print(f"  gene2 coding, gene1 not:  {only_g2:>8,}  ({only_g2/n*100:5.1f}%)")
    print(f"  involves a non-ENSG id:   {junk:>8,}  ({junk/n*100:5.1f}%)")
    print("\ncross-tab (rows = gene1, cols = gene2):")
    print(pd.crosstab(f.g1_status, f.g2_status, margins=True, margins_name="TOTAL").to_string())
    return f


fusions_status = fusion_coding_breakdown(fusions, coding_genes)

n_before = len(fusions)
fusions = (fusions_status.query("g1_status == 'coding' and g2_status == 'coding'")
                         .drop(columns=["g1_status", "g2_status"])
                         .reset_index(drop=True))
print(f"\nboth-coding filter: {n_before:,} -> {len(fusions):,} rows "
      f"({n_before - len(fusions):,} dropped)")
print(f"cell lines retained: {fusions.model_id.nunique():,}")

## 4. Identity hub — `cell_line_connection`

One row per ACH, with every alternative identifier held as a **list**. None of
these relationships is 1:1 — a model has several profiles (RNA, WES, WGS), a
line spans many GEO samples, and a CVCL can map to two ACHs (the `u-251 mg` case).

Holding lists keeps `model_id` unique, so every join to the hub stays one-to-one
and no count is inflated by duplication.

### 4.1 Backfill missing RRIDs via cellosaurus name match

Only rows with a missing `rrid` are touched — authoritative values are never
overwritten. Only unambiguous names (exactly one CVCL) are used as a source, so
a shared short-code never resolves arbitrarily.

In [ ]:
acc_by_name = (cellosaurus.dropna(subset=["cellosaurus_accession"])
                 .groupby("cellosaurus_cell_line_name")["cellosaurus_accession"]
                 .agg(lambda s: set(s.dropna())))
unambiguous_name = acc_by_name[acc_by_name.map(len) == 1].map(lambda s: next(iter(s)))

mask = sample_info["rrid"].isna()
before = mask.sum()
sample_info.loc[mask, "rrid"] = sample_info.loc[mask, "stripped_cell_line_name"].map(unambiguous_name)
after = sample_info["rrid"].isna().sum()
print(f"rrid missing: {before} -> filled: {before - after} -> still missing: {after}")

### 4.2 Cardinality checks

Run before any aggregation, to justify treating each identifier as a set rather
than a scalar. `model_id -> profile_id` is legitimately one-to-many (RNA/WES/WGS);
`cellosaurus_id -> geo_accession` likewise (a line profiled across several studies).

In [ ]:
report_cardinality(depmap_profiles, "profileid", "model_id", "depmap_profiles: profile <-> model")
report_cardinality(geo_info, "geo_accession", "cellosaurus_id", "geo_info: GSM <-> CVCL")
report_cardinality(hpa_desc, "cellosaurus id", "cell line", "hpa_desc: CVCL <-> cell line")
report_cardinality(metabolomics, "ccle_id", "model_id", "metabolomics: CCLE name <-> ACH")

### 4.3 Assemble the hub

The base is the **union of every model_id seen anywhere**, not `sample_info`.
`sample_info` is authoritative but not complete — `fusions` carries cell lines
that post-date its release, and starting from `sample_info` would make those
invisible. `in_roster` records the distinction.

Identifier axes that live on already-ACH-keyed tables are attached here; those
needing `model_id` from Section 5 are attached in 5.4.

In [ ]:
_sources = {"sample_info": sample_info, "depmap_profiles": depmap_profiles,
            "geo_info": geo_info, "metabolomics": metabolomics,
            "cellosaurus": cellosaurus, "hpa_desc": hpa_desc, "fusions": fusions,
            "geo_expr": geo_expr, "depmap_expr": depmap_expr, "hpa_rna": hpa_rna,
            "mutations": mutations, "proteomics": proteomics,
            "mirna": mirna, "signatures": signatures}

all_ids = set()
for _n, _df in _sources.items():
    if "model_id" in _df.columns:
        all_ids |= set(explode_ids(_df["model_id"]))

cell_line_connection = pd.DataFrame({"model_id": sorted(all_ids)})
cell_line_connection["in_roster"] = cell_line_connection["model_id"].isin(
    set(sample_info["model_id"].dropna()))
print(f"base: {len(cell_line_connection):,} ACHs "
      f"({cell_line_connection.in_roster.sum():,} in sample_info, "
      f"{(~cell_line_connection.in_roster).sum():,} outside it)")

for _src, _val, _name in [
    (sample_info,     "rrid",                    "rrids"),
    (depmap_profiles, "profileid",               "profile_ids"),
    (metabolomics,    "ccle_id",                 "ccle_ids"),
    (sample_info,     "cell_line_name",          "cell_line_names"),
    (sample_info,     "stripped_cell_line_name", "stripped_names"),
]:
    _s = collect(_src, _val, _name)
    if _s is not None:
        cell_line_connection = cell_line_connection.merge(_s, on="model_id", how="left")

# GEO accessions map via CVCL, then union across all of that ACH's CVCLs
geo_by_cvcl = (geo_info.dropna(subset=["cellosaurus_id"])
    .groupby("cellosaurus_id")["geo_accession"]
    .agg(lambda s: sorted(set(s.dropna()))))
geo_per_ach = (cell_line_connection[["model_id", "rrids"]].explode("rrids")
    .assign(geo=lambda d: d["rrids"].map(geo_by_cvcl))
    .groupby("model_id")["geo"].agg(geo_accessions=union_lists).reset_index())
cell_line_connection = cell_line_connection.merge(geo_per_ach, on="model_id", how="left")

# metabolomics is the ccle_id <-> ACH bridge (confirmed 1:1 in 4.2)
ccle_to_model = (metabolomics.dropna(subset=["model_id"])
                   .drop_duplicates("ccle_id").set_index("ccle_id")["model_id"])
mirna["model_id"] = mirna["ccle_name"].map(ccle_to_model)
_s = collect(mirna, "ccle_name", "ccle_names")
if _s is not None:
    cell_line_connection = cell_line_connection.merge(_s, on="model_id", how="left")

print(f"\nhub: {len(cell_line_connection):,} rows, model_id unique: "
      f"{cell_line_connection.model_id.is_unique}")

## 5. Attaching `model_id` to every table

All lookups derive from the hub via `build_lookup`, so there is one source of
truth per identifier axis. Every lookup returns a **list** — ambiguity stays
visible rather than being resolved arbitrarily by `drop_duplicates`.

In [ ]:
cvcl_to_models    = build_lookup(cell_line_connection, "rrids")
profile_to_models = build_lookup(cell_line_connection, "profile_ids")
gsm_to_models     = build_lookup(cell_line_connection, "geo_accessions")

name_to_models = (sample_info.dropna(subset=["model_id"])
                    .groupby("stripped_cell_line_name")["model_id"]
                    .agg(lambda s: sorted(set(s.dropna()))))

cellosaurus["model_id"] = cellosaurus["cellosaurus_accession"].map(cvcl_to_models)   # by CVCL
hpa_desc["model_id"]    = hpa_desc["cellosaurus id"].map(cvcl_to_models)

depmap_expr = ensure_key_column(depmap_expr, "profileid")                            # by profile
depmap_expr["model_id"] = depmap_expr["profileid"].map(profile_to_models)
mutations["model_id"]   = mutations["profileid"].map(profile_to_models)

geo_expr["model_id"] = geo_expr["sample"].map(gsm_to_models)                          # by GSM
geo_info["model_id"] = geo_info["cellline"].map(name_to_models)                       # by name

ATTACHED = {
    "hpa_rna": hpa_rna, "hpa_desc": hpa_desc, "depmap_expr": depmap_expr, "geo_expr": geo_expr,
    "proteomics": proteomics, "fusions": fusions, "mutations": mutations,
    "cellosaurus": cellosaurus, "depmap_profiles": depmap_profiles,
    "sample_info": sample_info, "geo_info": geo_info, "metabolomics": metabolomics,
    "mirna": mirna, "signatures": signatures,
}
for name, df in ATTACHED.items():
    if "model_id" not in df.columns:
        print(f"{name:18s} MISSING model_id")
    else:
        matched = df["model_id"].notna().sum()
        print(f"{name:18s} {matched:>9,} / {len(df):>9,} rows matched ({matched/len(df)*100:5.1f}%)")

### 5.1 Resolve ambiguity — separate candidate ACHs into their own rows

Rows whose CVCL or GSM resolves to more than one candidate ACH are **duplicated**,
one row per ACH. This is the *credit both* rule: the measurement genuinely belongs
to one of them and we don't know which, so crediting neither would understate
coverage for lines caught in an identity split.

`n_model_id` and `is_ambiguous` record that duplication happened, so any downstream
count can exclude them for a sensitivity check.

In [ ]:
print("exploding ambiguous model_id:")
cellosaurus = explode_model_id(cellosaurus, "cellosaurus")
hpa_desc    = explode_model_id(hpa_desc,    "hpa_desc")
geo_expr    = explode_model_id(geo_expr,    "geo_expr")
depmap_expr = explode_model_id(depmap_expr, "depmap_expr")
mutations   = explode_model_id(mutations,   "mutations")
geo_info    = explode_model_id(geo_info,    "geo_info")

# hpa_rna inherits model_id from hpa_desc, so it is mapped AFTER the explode via
# a merge. drop_duplicates() here would silently keep only the first ACH per cell
# line and discard the second — and is_ambiguous must come across too, or the
# duplicated rows become untraceable.
hpa_map = (hpa_desc.dropna(subset=["model_id"])[["cell line", "model_id", "is_ambiguous"]]
             .drop_duplicates())
n_before = len(hpa_rna)
hpa_rna = (hpa_rna.drop(columns=["model_id", "is_ambiguous"], errors="ignore")
                  .merge(hpa_map, on="cell line", how="left"))
print(f"  {'hpa_rna':18s} {n_before:>9,} -> {len(hpa_rna):>9,} rows  "
      f"(ambiguous {int(hpa_rna['is_ambiguous'].fillna(False).sum()):,})")

### 5.2 Unwrap to plain string

After exploding, each row holds one candidate — but it may still sit inside a
length-1 list. This normalises the dtype across every table, which is what lets
DuckDB join on `model_id` directly.

In [ ]:
print("unwrapping model_id to plain string:")
_all = {"hpa_rna": hpa_rna, "hpa_desc": hpa_desc, "depmap_expr": depmap_expr,
        "geo_expr": geo_expr, "proteomics": proteomics, "fusions": fusions,
        "mutations": mutations, "cellosaurus": cellosaurus,
        "depmap_profiles": depmap_profiles, "sample_info": sample_info,
        "geo_info": geo_info, "metabolomics": metabolomics,
        "mirna": mirna, "signatures": signatures}
_all = {n: unwrap_model_id(df, n) for n, df in _all.items()}
globals().update(_all)

### 5.3 Orphan check — records that don't trace back to a known ACH

Flagged, not dropped. `fusions` in particular contains cell lines with ACH ids
higher than anything in `sample_info` — a DepMap release-version gap, not a
pipeline fault.

In [ ]:
roster_achs     = set(explode_ids(cell_line_connection["model_id"]))
roster_profiles = set(explode_ids(cell_line_connection["profile_ids"]))

for label, ids, roster in [
        ("mutations (profileid)", set(mutations["profileid"].dropna()),   roster_profiles),
        ("fusions (model_id)",    set(fusions["model_id"].dropna()),      roster_achs),
        ("proteomics",            set(proteomics["model_id"].dropna()),   roster_achs),
        ("metabolomics",          set(metabolomics["model_id"].dropna()), roster_achs)]:
    missing = ids - roster
    print(f"{label:24s} {len(ids & roster):5d} / {len(ids):5d} in roster | orphaned: {len(missing)}")
    if missing:
        print(f"{'':24s} sample: {sorted(missing)[:5]}")

### 5.4 Back-fill the hub with identifiers from mapped tables

`cellosaurus` and `hpa_desc` only receive `model_id` in Section 5, via lookups
built *from* the hub — so their identifiers can't be collected in 4.3. They are
attached here, after the explode and unwrap, so the ambiguity they carry has
already been resolved into separate rows.

`cvcl_accessions` is an independent path to the same CVCL as `sample_info.rrid`;
where the two disagree, that's a genuine identity conflict worth inspecting.

In [ ]:
for _src, _val, _name in [
    (cellosaurus, "cellosaurus_accession", "cvcl_accessions"),
    (hpa_desc,    "cellosaurus id",        "hpa_cvcl_ids"),
    (hpa_desc,    "cell line",             "hpa_cell_lines"),
]:
    _s = collect(_src, _val, _name)
    if _s is None:
        continue
    cell_line_connection = cell_line_connection.merge(_s, on="model_id", how="left")

ID_COLS = ["rrids", "cvcl_accessions", "hpa_cvcl_ids", "profile_ids", "ccle_ids",
           "cell_line_names", "stripped_names", "hpa_cell_lines",
           "geo_accessions", "ccle_names"]
for c in ID_COLS:
    if c in cell_line_connection.columns:
        cell_line_connection[f"n_{c}"] = cell_line_connection[c].map(
            lambda x: len(x) if isinstance(x, LIST_LIKE) else 0)

print(f"\n{'identifier':20s} {'lines with it':>14s}")
for c in ID_COLS:
    if c in cell_line_connection.columns:
        print(f"{c:20s} {(cell_line_connection[f'n_{c}'] > 0).sum():>14,}")
print(f"\nACHs with >1 rrid (identity ambiguity): {(cell_line_connection['n_rrids'] > 1).sum()}")
cell_line_connection.head()

## 6. Coverage auditing

`depmap_expr` and `geo_expr` are kept as **separate** columns rather than collapsed
into one `gene_expression` flag. They are independent sources on the same axis and
can genuinely agree or disagree, which matters for the confidence score later.

In [ ]:
layer_ids = {name: set(explode_ids(globals()[name]["model_id"])) for name in ALL_LAYERS}
all_ids = set(explode_ids(cell_line_connection["model_id"]))

print(f"total model_id in hub: {len(all_ids):,}\n")
for name, ids in layer_ids.items():
    n = len(ids & all_ids)
    print(f"{name:18s} -> {n:5d} / {len(all_ids)} ({n/len(all_ids)*100:5.1f}%)")

coverage_matrix = pd.DataFrame({"model_id": sorted(all_ids)})
for name, ids in layer_ids.items():
    coverage_matrix[name] = coverage_matrix["model_id"].isin(ids)

coverage_matrix["n_modalities"] = coverage_matrix[MODALITY_LAYERS].sum(axis=1)
coverage_matrix["complete_cycle"] = coverage_matrix["n_modalities"] == len(MODALITY_LAYERS)

print(f"\nlines with all {len(MODALITY_LAYERS)} modalities: {coverage_matrix['complete_cycle'].sum()}")
print("\ndistribution of n_modalities:")
print(coverage_matrix["n_modalities"].value_counts().sort_index(ascending=False))
coverage_matrix.head()

### 6.1 Expression source breakdown

Which lines have DepMap, GEO, or both. The "both" set is where same-axis
corroboration is possible.

In [ ]:
depmap_in = layer_ids["depmap_expr"] & all_ids
geo_in    = layer_ids["geo_expr"] & all_ids

for label, s in [("depmap only", depmap_in - geo_in),
                 ("geo only",    geo_in - depmap_in),
                 ("both",        depmap_in & geo_in),
                 ("either",      depmap_in | geo_in)]:
    print(f"{label:12s} {len(s):5d} ({len(s)/len(all_ids)*100:5.1f}%)")

### 6.2 Gene axis coverage — ENSG overlap between expression sources

Recomputed after the protein-coding filter in 3.6. The intersection is the gene
universe over which DepMap and GEO can be compared.

In [ ]:
depmap_ensg = {c for c in depmap_expr.columns if str(c).startswith("ensg")}
geo_ensg    = {c for c in geo_expr.columns if str(c).startswith("ensg")}
common_ensg = sorted(depmap_ensg & geo_ensg)

print(f"ensg in depmap_expr: {len(depmap_ensg):,}")
print(f"ensg in geo_expr:    {len(geo_ensg):,}")
print(f"common to both:      {len(common_ensg):,}  "
      f"({len(common_ensg)/len(depmap_ensg)*100:.1f}% of depmap, "
      f"{len(common_ensg)/len(geo_ensg)*100:.1f}% of geo)")

## 7. Persist to DuckDB

Everything before this point is pandas; everything after is SQL against
`celllineselector.db`. `CREATE OR REPLACE` makes this cell safe to rerun.

The connection opened here stays open through Section 8, which builds the
dimension tables in SQL, and is closed once in Section 9.

In [ ]:
con = duckdb.connect(DB_PATH)

to_load = {**_all,
           "protein_map": protein_map,
           "cell_line_connection": cell_line_connection,
           "coverage_matrix": coverage_matrix}

for name, df in to_load.items():
    con.register("_tmp", df)
    con.execute(f'CREATE OR REPLACE TABLE "{name}" AS SELECT * FROM _tmp')
    con.unregister("_tmp")

print(f"wrote {len(to_load)} tables to {DB_PATH}")
print(con.execute("SHOW TABLES").df().to_string(index=False))

## 8. Gene dimension table

`cell_line_connection` (Section 4) is the cell line identity hub. This section
builds the equivalent for genes: one row per ENSG, alternative identifiers as
list columns, so a lookup by gene never has to know which layer holds what.


### 8.1 `gene` — every ENSG seen anywhere

Mirrors `cell_line_connection`: one row per gene, alternative identifiers held as
lists.

**Identifier axes**

- `gene_names` — a **list** of every distinct name seen for that ENSG across
  `hpa_rna`, `mutations` and both `fusions` partner columns. One ENSG legitimately
  carries several names through aliasing and release drift, so none is discarded.
  `NULL_TOKENS` (notably `.`, DepMap's null marker) are excluded.
- `hugo_symbol` / `hugo_symbol_all` — from `mutations` specifically
- `uniprot_ids` — joined from `protein_map` on symbol; a gene with several protein
  isoforms yields a list, keeping `gene_id` unique

`uniprot_ids` is what lets `proteomics` keep its original UniProt headers: the
gene→protein mapping lives here and is resolved at query time.

In [ ]:
def genes_from_long(con, table, gene_col, pattern=ENSG_PATTERN):
    """gene as a column VALUE, filtered to real ENSG ids and lowercased."""
    return set(con.execute(f'''
        SELECT DISTINCT lower(trim("{gene_col}")) AS g FROM "{table}"
        WHERE "{gene_col}" IS NOT NULL
          AND regexp_matches(lower(trim("{gene_col}")), '{pattern}')''').df()["g"])


def genes_from_wide(con, table, prefix="ensg"):
    """gene as a COLUMN NAME — depmap_expr, geo_expr."""
    cols = [r[0] for r in con.execute(f'DESCRIBE "{table}"').fetchall()]
    return {c for c in cols if c.lower().startswith(prefix)}


def genes_from_two_cols(con, table, col_a, col_b, pattern=ENSG_PATTERN):
    """fusions has two gene keys per row — both filtered, then unioned."""
    return (genes_from_long(con, table, col_a, pattern)
            | genes_from_long(con, table, col_b, pattern))


def symbol_lookup(con, table="mutations", gene_col="ensemblgeneid",
                  sym_col="hugosymbol", pattern=ENSG_PATTERN):
    """
    ENSG -> HUGO symbol. An ENSG can carry several symbols (alias drift across
    DepMap releases): hugo_symbol takes the first alphabetically,
    hugo_symbol_all keeps every variant so the choice stays inspectable.
    """
    d = con.execute(f'''
        SELECT DISTINCT lower(trim("{gene_col}")) AS gene_id,
                        trim("{sym_col}") AS symbol
        FROM "{table}"
        WHERE "{sym_col}" IS NOT NULL AND trim("{sym_col}") <> ''
          AND regexp_matches(lower(trim("{gene_col}")), '{pattern}')''').df()
    return (d.groupby("gene_id")["symbol"]
              .agg(hugo_symbol=lambda s: sorted(set(s))[0],
                   hugo_symbol_all=lambda s: "; ".join(sorted(set(s))),
                   n_hugo_symbols="nunique").reset_index())


def collect_gene_names(con, name_sources, pattern=ENSG_PATTERN):
    """
    ENSG -> sorted LIST of every distinct name seen for it, across all sources.
    One ENSG legitimately carries several names (aliases, release drift), so
    they are kept as a list rather than one being picked.

    Excluded: NULL_TOKENS ('.' is DepMap's null placeholder), and any value
    that is itself an ENSG id — some sources fall back to writing the ensembl
    id into the symbol column when no symbol is known, which would otherwise
    make a gene appear to be named after itself.
    """
    nulls = ", ".join(f"'{t}'" for t in NULL_TOKENS)
    parts = [f'SELECT lower(trim("{gc}")) AS gene_id, trim("{nc}") AS nm FROM "{t}"'
             for t, gc, nc in name_sources]
    return con.execute(f"""
        SELECT gene_id,
               list_sort(list_distinct(list(nm))) AS gene_names,
               count(DISTINCT nm)                 AS n_gene_names
        FROM ({" UNION ALL ".join(parts)})
        WHERE nm IS NOT NULL AND lower(nm) NOT IN ({nulls})
          AND NOT regexp_matches(lower(trim(nm)), '{pattern}')
          AND regexp_matches(gene_id, '{pattern}')
        GROUP BY gene_id""").df()

def build_gene_table(con, sets, name_sources, symbol_sources=None,
                     out="gene", pattern=ENSG_PATTERN,
                     uni_col="uniprot_id", sym_col="gene_symbol"):
    """
    Gene identity hub — one row per gene in the union of all source sets.

    name_sources: (table, gene_col, name_col, label) in priority order — the
    first source with a name wins. Genes where sources disagree are flagged in
    name_conflict rather than silently overwritten. The name lookup applies the
    SAME lower(trim()) and ENSG filter, or mixed-case ids fail to join.
    """
    g = pd.DataFrame({"gene_id": sorted(set.union(*sets.values()))})

    names = collect_gene_names(con, name_sources, pattern)
    g = g.merge(names, on="gene_id", how="left")
    g["n_gene_names"] = g["n_gene_names"].fillna(0).astype(int)
    # >1 name is normal (aliases); the flag marks it for inspection, not error
    g["name_conflict"] = g["n_gene_names"] > 1

    if symbol_sources:
        for table, gene_col, sc in symbol_sources:
            g = g.merge(symbol_lookup(con, table, gene_col, sc, pattern),
                        on="gene_id", how="left")
        g["hugo_symbol"] = g["hugo_symbol"].fillna("")
        g["hugo_symbol_all"] = g["hugo_symbol_all"].fillna("")
    else:
        g["hugo_symbol"] = ""
        g["hugo_symbol_all"] = ""

    g = g[["gene_id", "gene_names", "n_gene_names", "name_conflict",
           "hugo_symbol", "hugo_symbol_all"]]

    con.register("_gene_src", g)
    con.execute(f'''CREATE OR REPLACE TABLE "{out}" AS
        WITH up AS (
            SELECT lower(trim("{sym_col}")) AS sym,
                   list_sort(list_distinct(list(lower(trim("{uni_col}"))))) AS uniprot_ids
            FROM protein_map
            WHERE "{uni_col}" IS NOT NULL AND trim("{uni_col}") <> '' GROUP BY 1)
        SELECT s.gene_id, s.gene_names, s.n_gene_names, s.name_conflict,
               s.hugo_symbol, s.hugo_symbol_all,
               coalesce(up.uniprot_ids, [])      AS uniprot_ids,
               len(coalesce(up.uniprot_ids, [])) AS n_uniprot_ids
        FROM _gene_src s
        LEFT JOIN up ON lower(trim(s.hugo_symbol)) = up.sym
        ORDER BY s.gene_id''')
    con.unregister("_gene_src")

    d = con.execute(f'SELECT * FROM "{out}"').df()
    print(f"gene: {len(d):,} rows | gene_id unique: {d.gene_id.is_unique}")
    print(f"  with >=1 name:  {(d.n_gene_names > 0).sum():,}")
    print(f"  with >1 name:   {(d.n_gene_names > 1).sum():,}")
    print(f"  hugo_symbol:    {(d.hugo_symbol != '').sum():,}")
    print(f"  >=1 uniprot:    {(d.n_uniprot_ids > 0).sum():,}")
    print(f"  >1 uniprot:     {(d.n_uniprot_ids > 1).sum():,}")
    print(f"  name conflicts: {d.name_conflict.sum():,}")
    return d


gene_sets = {
    "hpa_rna":     genes_from_long(con, "hpa_rna", "gene"),
    "depmap_expr": genes_from_wide(con, "depmap_expr"),
    "geo_expr":    genes_from_wide(con, "geo_expr"),
    "mutations":   genes_from_long(con, "mutations", "ensemblgeneid"),
    "fusions":     genes_from_two_cols(con, "fusions", "gene1_ens_id", "gene2_ens_id"),
}
for n, s in gene_sets.items():
    print(f"  {n:14s} {len(s):>7,} genes")

gene = build_gene_table(
    con, gene_sets,
    name_sources=[("hpa_rna",   "gene",          "gene name"),
                  ("mutations", "ensemblgeneid", "hugosymbol"),
                  ("fusions",   "gene1_ens_id",  "gene1"),
                  ("fusions",   "gene2_ens_id",  "gene2")],
    symbol_sources=[("mutations", "ensemblgeneid", "hugosymbol")],
)
gene.head()

In [ ]:
# Genes with more than one gene name
gene_multiple_names = gene[gene["n_gene_names"] > 1]

print(f"Genes with >1 name: {len(gene_multiple_names)}")

print(gene_multiple_names[[
    "gene_id",
    "gene_names",
    "n_gene_names"
]])

### 8.2 Do fusion gene symbols agree with the reference names?

`fusions` carries a symbol (`gene1`, `gene2`) alongside each ENSG. This checks
whether that symbol is one of the names the reference sources give for the same
ENSG — a disagreement means the fusion caller and the annotation sources are
using different gene models.

**The reference set deliberately excludes fusions.** `gene_names` in 8.1 is built
*from* fusions among other sources, so comparing against it would be circular and
could never fail. Only `hpa_rna` and `mutations` are used here.

In [ ]:
def fusion_symbol_check(con, pattern=ENSG_PATTERN):
    """Compare each fusion partner's symbol against reference names for the same
    ENSG. Reference = hpa_rna + mutations only, so the check is independent of
    the fusion-derived names in gene.gene_names."""
    nulls = ", ".join(f"'{t}'" for t in NULL_TOKENS)
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE _refnames AS
        SELECT gene_id, list_sort(list_distinct(list(nm))) AS ref_names FROM (
            SELECT lower(trim(gene)) AS gene_id, trim("gene name") AS nm FROM hpa_rna
            UNION ALL
            SELECT lower(trim(ensemblgeneid)), trim(hugosymbol) FROM mutations)
        WHERE nm IS NOT NULL AND lower(nm) NOT IN ({nulls})
          AND regexp_matches(gene_id, '{pattern}')
        GROUP BY gene_id""")

    pairs = con.execute(f"""
        WITH p AS (
          SELECT model_id, 'gene1' AS side, trim(gene1) AS sym,
                 lower(trim(gene1_ens_id)) AS gene_id FROM fusions
          UNION ALL
          SELECT model_id, 'gene2', trim(gene2), lower(trim(gene2_ens_id)) FROM fusions)
        SELECT p.side, p.sym, p.gene_id, r.ref_names,
               CASE WHEN r.gene_id IS NULL THEN 'ensg has no reference name'
                    WHEN p.sym IS NULL OR lower(p.sym) IN ({nulls}) THEN 'no symbol in fusions'
                    WHEN list_contains(r.ref_names, p.sym) THEN 'match'
                    ELSE 'MISMATCH' END AS verdict
        FROM p LEFT JOIN _refnames r USING (gene_id)
        WHERE regexp_matches(p.gene_id, '{pattern}')""").df()

    print(f"fusion partner symbols checked: {len(pairs):,}")
    print(pairs.verdict.value_counts().to_string())
    mm = pairs[pairs.verdict == "MISMATCH"]
    if len(mm):
        print(f"\nsample mismatches ({len(mm):,}):")
        print(mm[["side", "sym", "gene_id", "ref_names"]].head(15).to_string(index=False))
    return pairs


fusion_symbols = fusion_symbol_check(con)

### 8.3 Orphaned UniProt accessions

Two classes worth separating:

- **A** — in `protein_map`, but its symbol matches no `hugo_symbol` in `gene`, so
  there's no route from that protein to a gene id
- **B** — a `proteomics` column absent from `protein_map` entirely, so no mapping
  information exists for that measurement at all

In [ ]:
def orphan_uniprots(con, gene_table="gene", map_table="protein_map",
                    prot_table="proteomics", uni_col="uniprot_id",
                    sym_col="gene_symbol", head=10):
    """Report UniProt ids that can't be resolved to a gene. Returns (df_A, df_B)."""
    a = con.execute(f'''
        SELECT lower(trim(p."{uni_col}")) AS uniprot_id,
               trim(p."{sym_col}")        AS gene_symbol,
               CASE WHEN p."{sym_col}" IS NULL OR trim(p."{sym_col}") = ''
                    THEN 'no symbol in protein_map'
                    ELSE 'symbol not in gene.hugo_symbol' END AS reason
        FROM "{map_table}" p
        LEFT JOIN "{gene_table}" g
          ON lower(trim(g.hugo_symbol)) = lower(trim(p."{sym_col}")) AND g.hugo_symbol <> ''
        WHERE p."{uni_col}" IS NOT NULL AND trim(p."{uni_col}") <> '' AND g.gene_id IS NULL
        ORDER BY 1''').df()
    total = con.execute(f'SELECT count(DISTINCT lower(trim("{uni_col}"))) '
                        f'FROM "{map_table}" WHERE "{uni_col}" IS NOT NULL').fetchone()[0]
    print(f"A) unattached in {map_table}: {len(a):,} of {total:,} ({len(a)/total*100:.1f}%)")
    if len(a):
        print(a.reason.value_counts().to_string())
        print(a.head(head).to_string(index=False))

    cols = [c for c in [r[0] for r in con.execute(f'DESCRIBE "{prot_table}"').fetchall()]
            if c != "model_id" and not str(c).lower().startswith("ensg")]
    known = set(con.execute(f'SELECT DISTINCT lower(trim("{uni_col}")) AS u '
                            f'FROM "{map_table}" WHERE "{uni_col}" IS NOT NULL').df()["u"])
    miss = [c for c in cols if str(c).strip().lower() not in known]
    b = pd.DataFrame({"proteomics_column": miss})
    print(f"\nB) proteomics columns absent from {map_table}: {len(miss):,} of {len(cols):,}")
    print(b.head(head).to_string(index=False) if miss else "  none")
    return a, b


orphan_a, orphan_b = orphan_uniprots(con)

## 9. Export for downstream stages

The DuckDB warehouse is the primary output, but Stage 1
(`01_lookup_metadata_join.ipynb`) and the Stage 2-7 scripts read parquet files
by path. These three exports keep that contract intact, so nothing downstream
has to learn SQL to keep working:

| file | contents | read by |
|---|---|---|
| `harmonised.parquet` | `cell_line_connection` — the identity hub | Stage 1 |
| `gene.parquet` | the gene dimension table (Section 8) | Stage 1 enrichment, `build_gene_roles.py` |
| `coverage_matrix.parquet` | per-line modality coverage | Stage 2 onwards, evidence ledger |

`harmonised.parquet` carries list-typed columns (`rrids`, `profile_ids`, …).
`pd.read_parquet` cannot round-trip `list<string>` back to a pandas dtype, which
is why Stage 1 reads it through `read_parquet_with_lists()`. That is unchanged
from the previous harmonisation — the column set is a superset of the old one,
so Stage 1 needs no edit.


In [ ]:
HARMONISED_PATH = os.path.join(OUT_DIR, "harmonised.parquet")
GENE_PATH       = os.path.join(OUT_DIR, "gene.parquet")
COVERAGE_PATH   = os.path.join(OUT_DIR, "coverage_matrix.parquet")

cell_line_connection.to_parquet(HARMONISED_PATH, index=False)
gene.to_parquet(GENE_PATH, index=False)
coverage_matrix.to_parquet(COVERAGE_PATH, index=False)

for label, path, df in [("harmonised    ", HARMONISED_PATH, cell_line_connection),
                        ("gene          ", GENE_PATH, gene),
                        ("coverage_matrix", COVERAGE_PATH, coverage_matrix)]:
    print(f"{label} {str(df.shape):>16s} -> {path}")

print("\nharmonised columns:", cell_line_connection.columns.tolist())


## 10. Verify the load

Reopens read-only and confirms every table landed with the expected shape.
This is the only place the connection is closed.

In [ ]:
con.close()
con = duckdb.connect(DB_PATH, read_only=True)

rows = []
for t in con.execute("SHOW TABLES").df()["name"]:
    cs = [r[0] for r in con.execute(f'DESCRIBE "{t}"').fetchall()]
    rows.append({
        "table": t,
        "rows": con.execute(f'SELECT count(*) FROM "{t}"').fetchone()[0],
        "cols": len(cs),
        "distinct_model_id": (con.execute(f'SELECT count(DISTINCT model_id) FROM "{t}"').fetchone()[0]
                              if "model_id" in cs else None),
        "gene_cols": len([c for c in cs if c.lower().startswith("ensg")]),
    })
print(pd.DataFrame(rows).sort_values("rows", ascending=False).to_string(index=False))

con.close()
print("\nconnection closed")